#### OVESAMPLING APPLIED TO 10 PERCENT

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from imblearn.over_sampling import ADASYN

# =============================
# Settings
# =============================
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour_bin']
TARGET_RATIO = 0.10  # 10% oversampling

# =============================
# Load dataset
# =============================
df = pd.read_csv("full_data_unhealthy_imputed_reduced_enhanced.csv")

# =============================
# Feature engineering
# =============================
def create_features(X):
    X = X.copy()
    X['hour_sin'] = np.sin(2 * np.pi * X['hour_bin'] / 24)
    X['hour_cos'] = np.cos(2 * np.pi * X['hour_bin'] / 24)
    X['eat_rest_ratio'] = X['EAT'] / (X['REST'] + 1e-6)
    X['activity_rest_ratio'] = np.abs(X['ACTIVITY_LEVEL']) / (X['REST'] + 1e-6)
    return X.drop(columns=['hour_bin'])

X = create_features(df[features])
y = df[targets]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =============================
# ADASYN per target
# =============================
def adasyn_resample_custom(X, y, target_ratio):
    X_res_list, y_res_list = [], []
    for target in targets:
        print(f"\nResampling target: {target}")
        counts = Counter(y[target])
        majority_class = counts[0]
        desired_minority = int(majority_class * target_ratio / (1 - target_ratio))
        print(f"Current counts: {counts}, Desired minority count: {desired_minority}")

        sampler = ADASYN(
            sampling_strategy={1: desired_minority},
            random_state=42,
            n_neighbors=5
        )
        X_res, y_res = sampler.fit_resample(X, y[target])

        X_res_list.append(pd.DataFrame(X_res, columns=X.columns))
        y_res_list.append(pd.Series(y_res, name=target))
    return X_res_list, y_res_list

# Perform oversampling
X_resampled_list, y_resampled_list = adasyn_resample_custom(X_train, y_train, TARGET_RATIO)

# Save each resampled dataset
for i, target in enumerate(targets):
    combined = pd.concat([X_resampled_list[i], y_resampled_list[i]], axis=1)
    out_file = f"adasyn_10pct_{target}.csv"
    combined.to_csv(out_file, index=False)
    print(f"Saved {out_file} with shape {combined.shape}")

print("\nOversampling complete. Datasets saved for each target.")



Resampling target: oestrus
Current counts: Counter({0: 253490, 1: 1042}), Desired minority count: 28165

Resampling target: calving
Current counts: Counter({0: 253961, 1: 571}), Desired minority count: 28217

Resampling target: lameness
Current counts: Counter({0: 254115, 1: 417}), Desired minority count: 28235

Resampling target: mastitis
Current counts: Counter({0: 254406, 1: 126}), Desired minority count: 28267
Saved adasyn_10pct_oestrus.csv with shape (281309, 9)
Saved adasyn_10pct_calving.csv with shape (282290, 9)
Saved adasyn_10pct_lameness.csv with shape (282380, 9)
Saved adasyn_10pct_mastitis.csv with shape (282702, 9)

Oversampling complete. Datasets saved for each target.


#### RANDOM FOREST AND LIGHT GBM APPLIED

In [3]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import classification_report, make_scorer, f1_score
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

# =============================
# Settings
# =============================
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
model_dir = "models/10pct"
os.makedirs(model_dir, exist_ok=True)

# =============================
# Scorer and CV setup
# =============================
scorer = make_scorer(lambda yt, yp: f1_score(yt, yp, average="weighted"))
cv = KFold(n_splits=3, shuffle=True, random_state=42)

# =============================
# Training function
# =============================
def tune_and_train(X_train, y_train, model_type):
    if model_type == "rf":
        base_model = RandomForestClassifier(random_state=42, n_jobs=-1)
        param_grid = {
            "n_estimators": [100, 150],
            "max_depth": [10, 20],
            "min_samples_split": [2, 5],
            "class_weight": ["balanced"]
        }
    else:
        base_model = LGBMClassifier(random_state=42, n_jobs=-1)
        param_grid = {
            "n_estimators": [200, 300],
            "max_depth": [12, 16],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8],
            "colsample_bytree": [0.8],
            "class_weight": ["balanced"]
        }

    grid = GridSearchCV(
        base_model,
        param_grid,
        scoring=scorer,
        cv=cv,
        verbose=2,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    print(f"Best params for {model_type.upper()}: {grid.best_params_}")
    return grid.best_estimator_

# =============================
# Train for each model and target
# =============================
for model_type in ["rf", "lgbm"]:
    print(f"\n=== Training {model_type.upper()} models with 10% ADASYN oversampling ===\n")
    for target in targets:
        print(f"--- {target} ---")
        df_res = pd.read_csv(f"adasyn_10pct_{target}.csv")
        X = df_res.drop(columns=[target])
        y = df_res[target]

        # Split into train/test
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        best_model = tune_and_train(X_train, y_train, model_type)

        # Evaluate
        y_pred = best_model.predict(X_val)
        print(f"\nClassification Report for {model_type.upper()} - {target}:\n")
        print(classification_report(y_val, y_pred))

        # Save model
        model_path = f"{model_dir}/{model_type}_{target}.pkl"
        joblib.dump(best_model, model_path)
        print(f"Saved model at {model_path}")



=== Training RF models with 10% ADASYN oversampling ===

--- oestrus ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best params for RF: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 150}

Classification Report for RF - oestrus:

              precision    recall  f1-score   support

           0       0.98      0.96      0.97     50784
           1       0.71      0.80      0.75      5478

    accuracy                           0.95     56262
   macro avg       0.84      0.88      0.86     56262
weighted avg       0.95      0.95      0.95     56262

Saved model at models/10pct/rf_oestrus.pkl
--- calving ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best params for RF: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 150}

Classification Report for RF - calving:

              precision    recall  f1-score   support

           0       0.99      0.98      0.98     50803
     

#### RE-SAMPLING

In [1]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import ADASYN
from sklearn.model_selection import train_test_split
from collections import Counter

# Load dataset
df = pd.read_csv("full_data_unhealthy_imputed_reduced_enhanced.csv")

# Features and targets
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour_bin']

def create_features(X):
    X = X.copy()
    X['hour_sin'] = np.sin(2 * np.pi * X['hour_bin'] / 24)
    X['hour_cos'] = np.cos(2 * np.pi * X['hour_bin'] / 24)
    X['eat_rest_ratio'] = X['EAT'] / (X['REST'] + 1e-6)
    X['activity_rest_ratio'] = np.abs(X['ACTIVITY_LEVEL']) / (X['REST'] + 1e-6)
    return X.drop(columns=['hour_bin'])

X = create_features(df[features])
y = df[targets]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Combined binary label
combined_target = (y_train.sum(axis=1) > 0).astype(int)

# Class distribution
counts = Counter(combined_target)
print("Original distribution:", counts)

# For 10% minority ratio
target_ratio = 0.10
majority = counts[0]
desired_minority = int(majority * target_ratio / (1 - target_ratio))
print("Desired minority:", desired_minority)

# Apply ADASYN
sampler = ADASYN(sampling_strategy={1: desired_minority}, random_state=42, n_neighbors=5)
X_res, combined_res = sampler.fit_resample(X_train, combined_target)

# Build final multi-label dataset
orig_len = len(X_train)
multi_y_res = []
for i in range(len(X_res)):
    if i < orig_len:
        multi_y_res.append(y_train.iloc[i].values)
    else:
        multi_y_res.append(np.random.multinomial(1, [0.25, 0.25, 0.25, 0.25]))

multi_y_res = np.array(multi_y_res)
y_res_df = pd.DataFrame(multi_y_res, columns=targets)
X_res_df = pd.DataFrame(X_res, columns=X.columns)

final_df = pd.concat([X_res_df, y_res_df], axis=1)
final_df.to_csv("adasyn_balanced_10pct.csv", index=False)
print("Saved adasyn_balanced_10pct.csv", final_df.shape)


Original distribution: Counter({0: 252388, 1: 2144})
Desired minority: 28043
Saved adasyn_balanced_10pct.csv (279680, 12)


#### RANDOM FOREST AND LIGHT GBM

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import classification_report, make_scorer, f1_score
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
import joblib
import os

# ==============================
# Configurations
# ==============================
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
model_dir = "models/10pct"
os.makedirs(model_dir, exist_ok=True)

# Load the 10% oversampled dataset
df = pd.read_csv("adasyn_balanced_10pct.csv")

# Define feature columns
feature_cols = ['IN_ALLEYS','REST','EAT','ACTIVITY_LEVEL',
                'hour_sin','hour_cos','eat_rest_ratio','activity_rest_ratio']

# Custom scorer (weighted F1)
scorer = make_scorer(lambda yt, yp: f1_score(yt, yp, average="weighted"))
cv = KFold(n_splits=3, shuffle=True, random_state=42)

# ==============================
# Function to train with GridSearch
# ==============================
def tune_and_train(X_train, y_train, model_type):
    if model_type == "rf":
        base_model = RandomForestClassifier(random_state=42, n_jobs=-1)
        param_grid = {
            "n_estimators": [100, 150],
            "max_depth": [10, 20],
            "min_samples_split": [2, 5],
            "class_weight": ["balanced"]
        }
    else:  # LightGBM
        base_model = LGBMClassifier(random_state=42, n_jobs=-1)
        param_grid = {
            "n_estimators": [200, 300],
            "max_depth": [12, 16],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8],
            "colsample_bytree": [0.8],
            "class_weight": ["balanced"]
        }

    grid = GridSearchCV(
        base_model,
        param_grid,
        scoring=scorer,
        cv=cv,
        verbose=2,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    print(f"Best params for {model_type.upper()}: {grid.best_params_}")
    return grid.best_estimator_

# ==============================
# Train and Save models
# ==============================
for model_type in ["rf", "lgbm"]:
    print(f"\n=== Training {model_type.upper()} models with 10% ADASYN oversampling ===\n")

    for target in targets:
        print(f"--- {target} ---")

        # Features and labels
        X = df[feature_cols]
        y = df[target]

        # Train/test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        # Train and tune
        best_model = tune_and_train(X_train, y_train, model_type)

        # Evaluate
        y_pred = best_model.predict(X_test)
        print(f"\nClassification Report for {model_type.upper()} - {target} (10% ADASYN):\n")
        print(classification_report(y_test, y_pred))

        # Save model
        model_path = f"{model_dir}/{model_type}_{target}_10pct.pkl"
        joblib.dump(best_model, model_path)
        print(f"Saved model at {model_path}")



=== Training RF models with 10% ADASYN oversampling ===

--- oestrus ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best params for RF: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 100}

Classification Report for RF - oestrus (10% ADASYN):

              precision    recall  f1-score   support

           0       0.98      0.97      0.97     54458
           1       0.23      0.37      0.28      1478

    accuracy                           0.95     55936
   macro avg       0.60      0.67      0.63     55936
weighted avg       0.96      0.95      0.96     55936

Saved model at models/10pct/rf_oestrus_10pct.pkl
--- calving ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best params for RF: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 150}

Classification Report for RF - calving (10% ADASYN):

              precision    recall  f1-score   support

           0       0.98    

#### MLP APPLIED

In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import os

# ==============================
# 1. Load dataset
# ==============================
df = pd.read_csv("adasyn_balanced_10pct.csv")  # Ensure this file exists

targets = ["oestrus", "calving", "lameness", "mastitis"]
features = [
    "IN_ALLEYS","REST","EAT","ACTIVITY_LEVEL",
    "hour_sin","hour_cos","eat_rest_ratio","activity_rest_ratio"
]

X = df[features].values
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================
# Dataset class
# ==============================
class MLPDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ==============================
# MLP Model
# ==============================
class MLP(nn.Module):
    def __init__(self, input_dim, hidden1=128, hidden2=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

# ==============================
# Training function
# ==============================
def train_model(model, train_loader, epochs=20, lr=0.001):
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb).view(-1)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss/len(train_loader):.4f}")
    return model

# ==============================
# Directory for saving
# ==============================
save_dir = "models/10pct_mlp"
os.makedirs(save_dir, exist_ok=True)

# ==============================
# Loop over targets
# ==============================
for target in targets:
    print(f"\n=== Training MLP for target: {target} (10% ADASYN) ===")
    
    y = df[target].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    train_ds = MLPDataset(X_train, y_train)
    test_ds = MLPDataset(X_test, y_test)
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)

    model = MLP(input_dim=X.shape[1]).to(device)
    model = train_model(model, train_loader, epochs=20, lr=0.001)

    # Evaluate
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in DataLoader(test_ds, batch_size=128):
            xb = xb.to(device)
            preds.append(model(xb).cpu().numpy())
    preds = (np.vstack(preds) > 0.5).astype(int)

    print(f"\nClassification report for MLP - {target}:\n")
    print(classification_report(y_test, preds, digits=4))

    # Save model
    model_path = os.path.join(save_dir, f"mlp_{target}_10pct.pth")
    torch.save(model.state_dict(), model_path)
    print(f"Model saved at {model_path}")



=== Training MLP for target: oestrus (10% ADASYN) ===
Epoch 1/20 Loss: 2.7223
Epoch 2/20 Loss: 2.6302
Epoch 3/20 Loss: 2.6302
Epoch 4/20 Loss: 2.6302
Epoch 5/20 Loss: 2.6302
Epoch 6/20 Loss: 2.6302
Epoch 7/20 Loss: 2.6302
Epoch 8/20 Loss: 2.6302
Epoch 9/20 Loss: 2.6302
Epoch 10/20 Loss: 2.6302
Epoch 11/20 Loss: 2.6302
Epoch 12/20 Loss: 2.6302
Epoch 13/20 Loss: 2.6307
Epoch 14/20 Loss: 2.6307
Epoch 15/20 Loss: 2.6302
Epoch 16/20 Loss: 2.6302
Epoch 17/20 Loss: 2.6302
Epoch 18/20 Loss: 2.6302
Epoch 19/20 Loss: 2.6302
Epoch 20/20 Loss: 2.6302

Classification report for MLP - oestrus:

              precision    recall  f1-score   support

           0     0.9736    1.0000    0.9866     54458
           1     0.0000    0.0000    0.0000      1478

    accuracy                         0.9736     55936
   macro avg     0.4868    0.5000    0.4933     55936
weighted avg     0.9479    0.9736    0.9605     55936

Model saved at models/10pct_mlp\mlp_oestrus_10pct.pth

=== Training MLP for target: 

C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/20 Loss: 2.4717
Epoch 2/20 Loss: 2.4667
Epoch 3/20 Loss: 2.4667
Epoch 4/20 Loss: 2.4667
Epoch 5/20 Loss: 2.4667
Epoch 6/20 Loss: 2.4667
Epoch 7/20 Loss: 2.4667
Epoch 8/20 Loss: 2.4667
Epoch 9/20 Loss: 2.4667
Epoch 10/20 Loss: 2.4667
Epoch 11/20 Loss: 2.4667
Epoch 12/20 Loss: 2.4665
Epoch 13/20 Loss: 2.4667
Epoch 14/20 Loss: 2.4667
Epoch 15/20 Loss: 2.4667
Epoch 16/20 Loss: 2.4667
Epoch 17/20 Loss: 2.4667
Epoch 18/20 Loss: 2.4667
Epoch 19/20 Loss: 2.4667
Epoch 20/20 Loss: 2.4667

Classification report for MLP - calving:

              precision    recall  f1-score   support

           0     0.9749    1.0000    0.9873     54530
           1     0.0000    0.0000    0.0000      1406

    accuracy                         0.9749     55936
   macro avg     0.4874    0.5000    0.4936     55936
weighted avg     0.9504    0.9749    0.9625     55936

Model saved at models/10pct_mlp\mlp_calving_10pct.pth

=== Training MLP for target: lameness (10% ADASYN) ===


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/20 Loss: 2.4806
Epoch 2/20 Loss: 2.3192
Epoch 3/20 Loss: 2.3192
Epoch 4/20 Loss: 2.3192
Epoch 5/20 Loss: 2.3190
Epoch 6/20 Loss: 2.3192
Epoch 7/20 Loss: 2.3192
Epoch 8/20 Loss: 2.3196
Epoch 9/20 Loss: 2.3192
Epoch 10/20 Loss: 2.3192
Epoch 11/20 Loss: 2.3192
Epoch 12/20 Loss: 2.3192
Epoch 13/20 Loss: 2.3192
Epoch 14/20 Loss: 2.3192
Epoch 15/20 Loss: 2.3192
Epoch 16/20 Loss: 2.3192
Epoch 17/20 Loss: 2.3192
Epoch 18/20 Loss: 2.3192
Epoch 19/20 Loss: 2.3192
Epoch 20/20 Loss: 2.3192

Classification report for MLP - lameness:

              precision    recall  f1-score   support

           0     0.9761    1.0000    0.9879     54597
           1     0.0000    0.0000    0.0000      1339

    accuracy                         0.9761     55936
   macro avg     0.4880    0.5000    0.4939     55936
weighted avg     0.9527    0.9761    0.9642     55936

Model saved at models/10pct_mlp\mlp_lameness_10pct.pth

=== Training MLP for target: mastitis (10% ADASYN) ===


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/20 Loss: 2.3403
Epoch 2/20 Loss: 2.3178
Epoch 3/20 Loss: 2.3178
Epoch 4/20 Loss: 2.3178
Epoch 5/20 Loss: 2.3178
Epoch 6/20 Loss: 2.3183
Epoch 7/20 Loss: 2.3178
Epoch 8/20 Loss: 2.3178
Epoch 9/20 Loss: 2.3178
Epoch 10/20 Loss: 2.3178
Epoch 11/20 Loss: 2.3178
Epoch 12/20 Loss: 2.3178
Epoch 13/20 Loss: 2.3178
Epoch 14/20 Loss: 2.3178
Epoch 15/20 Loss: 2.3178
Epoch 16/20 Loss: 2.3178
Epoch 17/20 Loss: 2.3174
Epoch 18/20 Loss: 2.3178
Epoch 19/20 Loss: 2.3178
Epoch 20/20 Loss: 2.3178

Classification report for MLP - mastitis:

              precision    recall  f1-score   support

           0     0.9767    1.0000    0.9882     54634
           1     0.0000    0.0000    0.0000      1302

    accuracy                         0.9767     55936
   macro avg     0.4884    0.5000    0.4941     55936
weighted avg     0.9540    0.9767    0.9652     55936

Model saved at models/10pct_mlp\mlp_mastitis_10pct.pth


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### TABNET APPLIED

In [11]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
import os

# ========================
# Config
# ========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = ['IN_ALLEYS','REST','EAT','ACTIVITY_LEVEL',
            'hour_sin','hour_cos','eat_rest_ratio','activity_rest_ratio']

save_dir = "models/10pct_mlp_improved"
os.makedirs(save_dir, exist_ok=True)

# ========================
# Load dataset
# ========================
df = pd.read_csv("adasyn_balanced_10pct.csv")  # ensure this dataset exists

# Normalize features
scaler = StandardScaler()
X_all = scaler.fit_transform(df[features])
np.save("models/10pct_mlp_improved/scaler.npy", scaler.mean_)

# ========================
# Model
# ========================
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

# Training function
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=50, patience=5):
    best_f1 = -1  # Initialize to -1 so that even 0.0 F1 is better
    wait = 0
    best_state = model.state_dict()  # Initialize to current weights

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb).squeeze()
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step(total_loss)

        # Validation
        model.eval()
        all_preds, all_true = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = (model(xb).squeeze() > 0.5).float()
                all_preds.extend(pred.cpu().numpy())
                all_true.extend(yb.cpu().numpy())

        from sklearn.metrics import f1_score
        f1 = f1_score(all_true, all_preds, zero_division=0)
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}, Val F1: {f1:.4f}")

        # Early stopping logic
        if f1 > best_f1:
            best_f1 = f1
            wait = 0
            best_state = model.state_dict()
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping.")
                break

    # Load the best model state
    model.load_state_dict(best_state)
    return model

# Train per target
# ========================
for target in targets:
    print(f"\n=== Training Improved MLP for {target} (10% ADASYN) ===")
    y_all = df[target].values

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=0.2, random_state=42
    )

    # Convert to PyTorch tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.float32)

    # DataLoader
    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=128, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=128)

    # Model, loss, optimizer
    model = MLP(X_train.shape[1]).to(device)

    # Compute class weights (to handle imbalance)
    pos_weight = (len(y_train) - sum(y_train)) / (sum(y_train) + 1e-6)
    criterion = nn.BCELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

    # Train
    model = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler)

    # Evaluate
    model.eval()
    with torch.no_grad():
        preds = (model(X_test_t.to(device)).squeeze() > 0.5).cpu().numpy()
    print(classification_report(y_test, preds, digits=4))

    # Save model
    torch.save(model.state_dict(), os.path.join(save_dir, f"mlp_{target}_10pct.pth"))
    print(f"Saved model: {save_dir}/mlp_{target}_10pct.pth")



=== Training Improved MLP for oestrus (10% ADASYN) ===
Epoch 1, Loss: 0.1210, Val F1: 0.0000
Epoch 2, Loss: 0.1096, Val F1: 0.0000
Epoch 3, Loss: 0.1074, Val F1: 0.0000
Epoch 4, Loss: 0.1057, Val F1: 0.0000
Epoch 5, Loss: 0.1049, Val F1: 0.0000
Epoch 6, Loss: 0.1038, Val F1: 0.0000
Early stopping.
              precision    recall  f1-score   support

           0     0.9736    1.0000    0.9866     54458
           1     0.0000    0.0000    0.0000      1478

    accuracy                         0.9736     55936
   macro avg     0.4868    0.5000    0.4933     55936
weighted avg     0.9479    0.9736    0.9605     55936

Saved model: models/10pct_mlp_improved/mlp_oestrus_10pct.pth

=== Training Improved MLP for calving (10% ADASYN) ===


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1, Loss: 0.1140, Val F1: 0.0000
Epoch 2, Loss: 0.1007, Val F1: 0.0000
Epoch 3, Loss: 0.0998, Val F1: 0.0000
Epoch 4, Loss: 0.0985, Val F1: 0.0000
Epoch 5, Loss: 0.0975, Val F1: 0.0000
Epoch 6, Loss: 0.0972, Val F1: 0.0000
Early stopping.
              precision    recall  f1-score   support

           0     0.9749    1.0000    0.9873     54530
           1     0.0000    0.0000    0.0000      1406

    accuracy                         0.9749     55936
   macro avg     0.4874    0.5000    0.4936     55936
weighted avg     0.9504    0.9749    0.9625     55936

Saved model: models/10pct_mlp_improved/mlp_calving_10pct.pth

=== Training Improved MLP for lameness (10% ADASYN) ===


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1, Loss: 0.1091, Val F1: 0.0000
Epoch 2, Loss: 0.0977, Val F1: 0.0000
Epoch 3, Loss: 0.0956, Val F1: 0.0000
Epoch 4, Loss: 0.0947, Val F1: 0.0000
Epoch 5, Loss: 0.0940, Val F1: 0.0000
Epoch 6, Loss: 0.0931, Val F1: 0.0000
Early stopping.
              precision    recall  f1-score   support

           0     0.9761    1.0000    0.9879     54597
           1     0.0000    0.0000    0.0000      1339

    accuracy                         0.9761     55936
   macro avg     0.4880    0.5000    0.4939     55936
weighted avg     0.9527    0.9761    0.9642     55936

Saved model: models/10pct_mlp_improved/mlp_lameness_10pct.pth

=== Training Improved MLP for mastitis (10% ADASYN) ===


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1, Loss: 0.1102, Val F1: 0.0000
Epoch 2, Loss: 0.0975, Val F1: 0.0000
Epoch 3, Loss: 0.0955, Val F1: 0.0000
Epoch 4, Loss: 0.0940, Val F1: 0.0000
Epoch 5, Loss: 0.0931, Val F1: 0.0000
Epoch 6, Loss: 0.0923, Val F1: 0.0000
Early stopping.
              precision    recall  f1-score   support

           0     0.9767    1.0000    0.9882     54634
           1     0.0000    0.0000    0.0000      1302

    accuracy                         0.9767     55936
   macro avg     0.4884    0.5000    0.4941     55936
weighted avg     0.9540    0.9767    0.9652     55936

Saved model: models/10pct_mlp_improved/mlp_mastitis_10pct.pth


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### TABNET APPLIED

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from pytorch_tabnet.tab_model import TabNetClassifier
import torch
import os

# =============================
# 1. Load dataset
# =============================
df = pd.read_csv("adasyn_balanced_10pct.csv")

targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = [
    'IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL',
    'hour_sin', 'hour_cos', 'eat_rest_ratio', 'activity_rest_ratio'
]

X = df[features].values
y = df[targets].values

# Output folder
save_dir = "models/10pct_tabnet"
os.makedirs(save_dir, exist_ok=True)

# =============================
# 2. Train/test split
# =============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =============================
# 3. Train TabNet for each target
# =============================
for idx, target in enumerate(targets):
    print(f"\n=== Training TabNet for target: {target} (10% ADASYN) ===")

    y_train_target = y_train[:, idx]
    y_test_target = y_test[:, idx]

    clf = TabNetClassifier(
        n_d=16, n_a=16,
        n_steps=3,
        gamma=1.5,
        lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2),
        scheduler_params={"step_size": 50, "gamma": 0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        seed=42,
        verbose=10
    )

    # Fit model
    clf.fit(
        X_train, y_train_target,
        eval_set=[(X_test, y_test_target)],
        eval_name=['valid'],
        eval_metric=['balanced_accuracy'],
        max_epochs=50,
        patience=10,
        batch_size=2048,
        virtual_batch_size=256,
        num_workers=0,
        drop_last=False
    )

    # Evaluate
    preds = clf.predict(X_test)
    print(f"\nClassification report for TabNet - {target}:\n")
    print(classification_report(y_test_target, preds, digits=4))

    # Save model
    save_path = f"{save_dir}/tabnet_{target}_10pct.zip"
    clf.save_model(save_path)
    print(f"Model saved at {save_path}")



=== Training TabNet for target: oestrus (10% ADASYN) ===


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.12734 | valid_balanced_accuracy: 0.5003  |  0:00:12s
epoch 10 | loss: 0.09898 | valid_balanced_accuracy: 0.48685 |  0:02:17s

Early stopping occurred at epoch 13 with best_epoch = 3 and best_valid_balanced_accuracy = 0.50376


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Classification report for TabNet - oestrus:

              precision    recall  f1-score   support

           0     0.9738    0.9994    0.9864     54458
           1     0.2667    0.0081    0.0158      1478

    accuracy                         0.9732     55936
   macro avg     0.6202    0.5038    0.5011     55936
weighted avg     0.9551    0.9732    0.9608     55936

Successfully saved model at models/10pct_tabnet/tabnet_oestrus_10pct.zip.zip
Model saved at models/10pct_tabnet/tabnet_oestrus_10pct.zip

=== Training TabNet for target: calving (10% ADASYN) ===


...


Classification report for TabNet - lameness:

              precision    recall  f1-score   support

           0     0.9761    0.9995    0.9877     54597
           1     0.1071    0.0022    0.0044      1339

    accuracy                         0.9757     55936
   macro avg     0.5416    0.5009    0.4960     55936
weighted avg     0.9553    0.9757    0.9641     55936

Successfully saved model at models/10pct_tabnet/tabnet_lameness_10pct.zip.zip
Model saved at models/10pct_tabnet/tabnet_lameness_10pct.zip

=== Training TabNet for target: mastitis (10% ADASYN) ===


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.11224 | valid_balanced_accuracy: 0.50058 |  0:00:14s
epoch 10 | loss: 0.08755 | valid_balanced_accuracy: 0.50037 |  0:02:26s

Early stopping occurred at epoch 10 with best_epoch = 0 and best_valid_balanced_accuracy = 0.50058


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Classification report for TabNet - mastitis:

              precision    recall  f1-score   support

           0     0.9767    0.9996    0.9881     54634
           1     0.0870    0.0015    0.0030      1302

    accuracy                         0.9764     55936
   macro avg     0.5319    0.5006    0.4955     55936
weighted avg     0.9560    0.9764    0.9651     55936

Successfully saved model at models/10pct_tabnet/tabnet_mastitis_10pct.zip.zip
Model saved at models/10pct_tabnet/tabnet_mastitis_10pct.zip


#### LSTM APPLIED

In [15]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import os

# ==================================================
# Load dataset (10% ADASYN)
# ==================================================
df = pd.read_csv("adasyn_balanced_10pct.csv")

targets = ["oestrus", "calving", "lameness", "mastitis"]
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL']

SEQ_LEN = 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Directory to save models
save_dir = "models/10pct_lstm"
os.makedirs(save_dir, exist_ok=True)

# ==================================================
# Create temporal sequences
# ==================================================
def create_sequences(data, labels, seq_length=SEQ_LEN):
    X_seq, y_seq = [], []
    for i in range(len(data) - seq_length + 1):
        X_seq.append(data[i:i+seq_length])
        y_seq.append(labels[i+seq_length-1])
    return np.array(X_seq), np.array(y_seq)

# Dataset class
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ==================================================
# LSTM Model
# ==================================================
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return torch.sigmoid(out)

# ==================================================
# Training function
# ==================================================
def train_model(model, train_loader, epochs=20, lr=0.001):
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs.view(-1), y_batch.view(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")
    return model

# ==================================================
# Train models per target
# ==================================================
results = {}
X_all = df[features].values

for target in targets:
    print(f"\n=== Training LSTM for target: {target} (10% ADASYN) ===")
    y_all = df[target].values
    X_seq, y_seq = create_sequences(X_all, y_all)

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)

    # Data loaders
    train_dataset = SeqDataset(X_train, y_train)
    test_dataset = SeqDataset(X_test, y_test)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

    # Model
    model = LSTMModel(input_dim=X_train.shape[2]).to(device)
    model = train_model(model, train_loader, epochs=20, lr=0.001)

    # Evaluate
    model.eval()
    preds = []
    with torch.no_grad():
        for X_batch, _ in DataLoader(test_dataset, batch_size=64):
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            preds.append(outputs.cpu().numpy())
    preds = (np.vstack(preds) > 0.5).astype(int)

    print(f"\nClassification report for LSTM - {target}:\n")
    print(classification_report(y_test, preds, digits=4))

    # Save results and model
    results[target] = classification_report(y_test, preds, digits=4, output_dict=True)
    model_path = os.path.join(save_dir, f"lstm_{target}_10pct.pth")
    torch.save(model.state_dict(), model_path)
    print(f"Saved LSTM model for {target} at {model_path}")

print("\nAll LSTM models (10% ADASYN) trained and saved successfully.")



=== Training LSTM for target: oestrus (10% ADASYN) ===
Epoch 1/20, Loss: 0.1062
Epoch 2/20, Loss: 0.0968
Epoch 3/20, Loss: 0.0951
Epoch 4/20, Loss: 0.0944
Epoch 5/20, Loss: 0.0953
Epoch 6/20, Loss: 0.0960
Epoch 7/20, Loss: 0.0956
Epoch 8/20, Loss: 0.0945
Epoch 9/20, Loss: 0.0953
Epoch 10/20, Loss: 0.0959
Epoch 11/20, Loss: 0.0957
Epoch 12/20, Loss: 0.0964
Epoch 13/20, Loss: 0.0967
Epoch 14/20, Loss: 0.0956
Epoch 15/20, Loss: 0.0954
Epoch 16/20, Loss: 0.0960
Epoch 17/20, Loss: 0.0955
Epoch 18/20, Loss: 0.0954
Epoch 19/20, Loss: 0.0955
Epoch 20/20, Loss: 0.0954

Classification report for LSTM - oestrus:

              precision    recall  f1-score   support

           0     0.9727    0.9998    0.9861     54406
           1     0.3125    0.0033    0.0065      1530

    accuracy                         0.9725     55936
   macro avg     0.6426    0.5015    0.4963     55936
weighted avg     0.9547    0.9725    0.9593     55936

Saved LSTM model for oestrus at models/10pct_lstm\lstm_oestrus

C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packag

Epoch 1/20, Loss: 0.0981
Epoch 2/20, Loss: 0.0869
Epoch 3/20, Loss: 0.0849
Epoch 4/20, Loss: 0.0843
Epoch 5/20, Loss: 0.0834
Epoch 6/20, Loss: 0.0837
Epoch 7/20, Loss: 0.0837
Epoch 8/20, Loss: 0.0827
Epoch 9/20, Loss: 0.0832
Epoch 10/20, Loss: 0.0840
Epoch 11/20, Loss: 0.0849
Epoch 12/20, Loss: 0.0853
Epoch 13/20, Loss: 0.0843
Epoch 14/20, Loss: 0.0845
Epoch 15/20, Loss: 0.0853
Epoch 16/20, Loss: 0.0847
Epoch 17/20, Loss: 0.0839
Epoch 18/20, Loss: 0.0857
Epoch 19/20, Loss: 0.0849
Epoch 20/20, Loss: 0.0863

Classification report for LSTM - lameness:

              precision    recall  f1-score   support

           0     0.9769    1.0000    0.9883     54646
           1     0.0000    0.0000    0.0000      1290

    accuracy                         0.9769     55936
   macro avg     0.4885    0.5000    0.4942     55936
weighted avg     0.9544    0.9769    0.9655     55936

Saved LSTM model for lameness at models/10pct_lstm\lstm_lameness_10pct.pth

=== Training LSTM for target: mastitis (1

C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packag

Epoch 1/20, Loss: 0.0962
Epoch 2/20, Loss: 0.0835
Epoch 3/20, Loss: 0.0823
Epoch 4/20, Loss: 0.0825
Epoch 5/20, Loss: 0.0825
Epoch 6/20, Loss: 0.0810
Epoch 7/20, Loss: 0.0814
Epoch 8/20, Loss: 0.0803
Epoch 9/20, Loss: 0.0810
Epoch 10/20, Loss: 0.0816
Epoch 11/20, Loss: 0.0808
Epoch 12/20, Loss: 0.0807
Epoch 13/20, Loss: 0.0809
Epoch 14/20, Loss: 0.0841
Epoch 15/20, Loss: 0.0822
Epoch 16/20, Loss: 0.0812
Epoch 17/20, Loss: 0.0823
Epoch 18/20, Loss: 0.0824
Epoch 19/20, Loss: 0.0818
Epoch 20/20, Loss: 0.0819

Classification report for LSTM - mastitis:

              precision    recall  f1-score   support

           0     0.9773    0.9986    0.9878     54648
           1     0.2277    0.0179    0.0331      1288

    accuracy                         0.9760     55936
   macro avg     0.6025    0.5082    0.5105     55936
weighted avg     0.9601    0.9760    0.9659     55936

Saved LSTM model for mastitis at models/10pct_lstm\lstm_mastitis_10pct.pth

All LSTM models (10% ADASYN) trained and 

#### TCN APPLIED

In [29]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import os

# ===============================
# Setup
# ===============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEQ_LEN = 3

# Load dataset (10% ADASYN oversampled)
df = pd.read_csv("adasyn_balanced_10pct.csv")

# Define features and targets
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
feature_cols = [
    'IN_ALLEYS','REST','EAT','ACTIVITY_LEVEL',
    'hour_sin','hour_cos','eat_rest_ratio','activity_rest_ratio'
]

# Directory for saving TCN models
tcn_save_dir = "models/10pct_tcn"
os.makedirs(tcn_save_dir, exist_ok=True)

# ===============================
# Dataset utilities
# ===============================
def create_sequences(data, target_col):
    X, y = [], []
    for i in range(len(data) - SEQ_LEN):
        X.append(data.iloc[i:i+SEQ_LEN][feature_cols].values)
        y.append(data.iloc[i+SEQ_LEN][target_col])
    return np.array(X), np.array(y)

class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ===============================
# TCN model definition
# ===============================
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size]

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout):
        super().__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(
            self.conv1, self.chomp1, self.relu1, self.dropout1,
            self.conv2, self.chomp2, self.relu2, self.dropout2
        )
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNModel(nn.Module):
    def __init__(self, input_dim, output_dim, num_channels=[16, 16], kernel_size=2, dropout=0.25):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation_size = 2 ** i
            in_channels = input_dim if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(
                in_channels, out_channels, kernel_size, stride=1,
                dilation=dilation_size,
                padding=(kernel_size-1)*dilation_size,
                dropout=dropout
            )]
        self.network = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.permute(0, 2, 1)  # [batch, features, seq_len]
        y = self.network(x)
        y = y[:, :, -1]
        y = self.fc(y)
        return self.sigmoid(y)

# ===============================
# Training function
# ===============================
def train_tcn(X_train, y_train, input_dim, epochs=15, batch_size=128):
    dataset = SequenceDataset(X_train, y_train)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = TCNModel(input_dim=input_dim, output_dim=1).to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb).squeeze()
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(loader):.4f}")
    return model

# ===============================
# Training and saving per target
# ===============================
for target in targets:
    print(f"\n=== Training TCN for target: {target} (10% ADASYN) ===")
    X, y = create_sequences(df, target)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = train_tcn(X_train, y_train, input_dim=X.shape[2], epochs=15)

    # Save model
    save_path = os.path.join(tcn_save_dir, f"tcn_{target}_10pct.pth")
    torch.save(model.state_dict(), save_path)
    print(f"Saved TCN model for {target} at: {save_path}")

    # Evaluation
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X_test, dtype=torch.float32, device=device)).squeeze().cpu().numpy()
    preds_bin = (preds > 0.5).astype(int)

    print(f"\nClassification report for TCN - {target}:\n")
    print(classification_report(y_test, preds_bin, digits=4))



=== Training TCN for target: oestrus (10% ADASYN) ===
Epoch 1/15, Loss: 3.4423
Epoch 2/15, Loss: 2.7433
Epoch 3/15, Loss: 2.6240
Epoch 4/15, Loss: 2.6227
Epoch 5/15, Loss: 2.6289
Epoch 6/15, Loss: 2.6236
Epoch 7/15, Loss: 2.6272
Epoch 8/15, Loss: 2.6285
Epoch 9/15, Loss: 2.6303
Epoch 10/15, Loss: 2.6254
Epoch 11/15, Loss: 2.6258
Epoch 12/15, Loss: 2.6267
Epoch 13/15, Loss: 2.6249
Epoch 14/15, Loss: 2.6289
Epoch 15/15, Loss: 2.6240
Saved TCN model for oestrus at: models/10pct_tcn\tcn_oestrus_10pct.pth

Classification report for TCN - oestrus:

              precision    recall  f1-score   support

         0.0     0.9730    1.0000    0.9863     54425
         1.0     0.0000    0.0000    0.0000      1511

    accuracy                         0.9730     55936
   macro avg     0.4865    0.5000    0.4932     55936
weighted avg     0.9467    0.9730    0.9597     55936


=== Training TCN for target: calving (10% ADASYN) ===


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/15, Loss: 97.2223
Epoch 2/15, Loss: 97.2214
Epoch 3/15, Loss: 97.2289
Epoch 4/15, Loss: 97.2357
Epoch 5/15, Loss: 97.2124
Epoch 6/15, Loss: 97.1815
Epoch 7/15, Loss: 33.8536
Epoch 8/15, Loss: 2.5293
Epoch 9/15, Loss: 2.5261
Epoch 10/15, Loss: 2.5203
Epoch 11/15, Loss: 2.5337
Epoch 12/15, Loss: 2.5306
Epoch 13/15, Loss: 2.5279
Epoch 14/15, Loss: 2.5233
Epoch 15/15, Loss: 2.5212
Saved TCN model for calving at: models/10pct_tcn\tcn_calving_10pct.pth

Classification report for TCN - calving:

              precision    recall  f1-score   support

         0.0     0.9756    1.0000    0.9876     54570
         1.0     0.0000    0.0000    0.0000      1366

    accuracy                         0.9756     55936
   macro avg     0.4878    0.5000    0.4938     55936
weighted avg     0.9518    0.9756    0.9635     55936


=== Training TCN for target: lameness (10% ADASYN) ===


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/15, Loss: 3.9317
Epoch 2/15, Loss: 2.3335
Epoch 3/15, Loss: 2.3339
Epoch 4/15, Loss: 2.3335
Epoch 5/15, Loss: 2.3335
Epoch 6/15, Loss: 2.3335
Epoch 7/15, Loss: 2.3335
Epoch 8/15, Loss: 2.3335
Epoch 9/15, Loss: 2.3335
Epoch 10/15, Loss: 2.3335
Epoch 11/15, Loss: 2.3335
Epoch 12/15, Loss: 2.3335
Epoch 13/15, Loss: 2.3335
Epoch 14/15, Loss: 2.3335
Epoch 15/15, Loss: 2.3335
Saved TCN model for lameness at: models/10pct_tcn\tcn_lameness_10pct.pth

Classification report for TCN - lameness:

              precision    recall  f1-score   support

         0.0     0.9766    1.0000    0.9882     54629
         1.0     0.0000    0.0000    0.0000      1307

    accuracy                         0.9766     55936
   macro avg     0.4883    0.5000    0.4941     55936
weighted avg     0.9538    0.9766    0.9651     55936


=== Training TCN for target: mastitis (10% ADASYN) ===


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/15, Loss: 2.9922
Epoch 2/15, Loss: 2.3295
Epoch 3/15, Loss: 2.3299
Epoch 4/15, Loss: 2.3308
Epoch 5/15, Loss: 2.3254
Epoch 6/15, Loss: 2.3249
Epoch 7/15, Loss: 2.3280
Epoch 8/15, Loss: 2.3291
Epoch 9/15, Loss: 2.3317
Epoch 10/15, Loss: 2.3291
Epoch 11/15, Loss: 2.3262
Epoch 12/15, Loss: 2.3331
Epoch 13/15, Loss: 2.3161
Epoch 14/15, Loss: 2.3049
Epoch 15/15, Loss: 2.3049
Saved TCN model for mastitis at: models/10pct_tcn\tcn_mastitis_10pct.pth

Classification report for TCN - mastitis:

              precision    recall  f1-score   support

         0.0     0.9760    1.0000    0.9879     54596
         1.0     0.0000    0.0000    0.0000      1340

    accuracy                         0.9760     55936
   macro avg     0.4880    0.5000    0.4939     55936
weighted avg     0.9527    0.9760    0.9642     55936



C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### HYBRID ENSEMBLE METHOD APPLIED

In [31]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import joblib
import os

# ---------------------------
# CONFIGURATION
# ---------------------------
SEQ_LEN = 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Directories for saved models
rf_dir = "models/10pct"
lgbm_dir = "models/10pct"
lstm_dir = "models/10pct_lstm"
tcn_dir = "models/10pct_tcn"

# Target columns
targets = ["oestrus", "calving", "lameness", "mastitis"]

# Features
features_4 = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL']  # for LSTM
features_8 = [
    'IN_ALLEYS','REST','EAT','ACTIVITY_LEVEL',
    'hour_sin','hour_cos','eat_rest_ratio','activity_rest_ratio'
]  # for RF, LGBM, TCN

# ---------------------------
# Dataset and model classes
# ---------------------------
class SeqDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx]

def create_sequences(data, labels, seq_len=3):
    X_seq, y_seq = [], []
    for i in range(len(data) - seq_len + 1):
        X_seq.append(data[i:i+seq_len])
        y_seq.append(labels[i+seq_len-1])
    return np.array(X_seq), np.array(y_seq)

# LSTM model
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return torch.sigmoid(out)

# TCN components
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size]

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout):
        super().__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net = nn.Sequential(
            self.conv1, self.chomp1, self.relu1, self.dropout1,
            self.conv2, self.chomp2, self.relu2, self.dropout2
        )
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNModel(nn.Module):
    def __init__(self, input_dim, output_dim, num_channels=[16, 16], kernel_size=2, dropout=0.25):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation_size = 2 ** i
            in_channels = input_dim if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size,
                                     stride=1, dilation=dilation_size,
                                     padding=(kernel_size-1)*dilation_size,
                                     dropout=dropout)]
        self.network = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], output_dim)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = x.permute(0, 2, 1)
        y = self.network(x)
        y = y[:, :, -1]
        y = self.fc(y)
        return self.sigmoid(y)

# ---------------------------
# Hybrid ensemble
# ---------------------------
# Load dataset (10% ADASYN balanced)
df = pd.read_csv("adasyn_balanced_10pct.csv")

# Prepare features
X_4 = df[features_4]
X_8 = df[features_8]

# Loop for each target
for target in targets:
    print(f"\n=== Hybrid Ensemble for target: {target} (10% ADASYN) ===")

    y = df[target]

    # Split for tabular models (RF, LGBM)
    X_train_8, X_test_8, y_train, y_test = train_test_split(
        X_8, y, test_size=0.2, random_state=42
    )

    # Create sequences
    # For LSTM: use 4 features
    X_seq_4, y_seq_4 = create_sequences(X_4.values, y.values, SEQ_LEN)
    _, X_seq_test_4 = train_test_split(X_seq_4, test_size=0.2, random_state=42)

    # For TCN: use 8 features
    X_seq_8, y_seq_8 = create_sequences(X_8.values, y.values, SEQ_LEN)
    _, X_seq_test_8 = train_test_split(X_seq_8, test_size=0.2, random_state=42)

    # ---- Load models ----
    rf = joblib.load(f"{rf_dir}/rf_{target}_10pct.pkl")
    lgbm = joblib.load(f"{lgbm_dir}/lgbm_{target}_10pct.pkl")

    lstm = LSTMModel(input_dim=X_seq_test_4.shape[2])
    lstm.load_state_dict(torch.load(f"{lstm_dir}/lstm_{target}_10pct.pth", map_location=device))
    lstm.to(device).eval()

    tcn = TCNModel(input_dim=X_seq_test_8.shape[2], output_dim=1)
    tcn.load_state_dict(torch.load(f"{tcn_dir}/tcn_{target}_10pct.pth", map_location=device))
    tcn.to(device).eval()

    # ---- Predictions ----
    # RF + LGBM (tabular, 8 features)
    rf_probs = rf.predict_proba(X_test_8)[:, 1]
    lgbm_probs = lgbm.predict_proba(X_test_8)[:, 1]

    # LSTM (4 features)
    lstm_dataset = SeqDataset(X_seq_test_4)
    lstm_loader = DataLoader(lstm_dataset, batch_size=128)
    lstm_probs = []
    with torch.no_grad():
        for xb in lstm_loader:
            xb = xb.to(device)
            lstm_probs.append(lstm(xb).cpu().numpy())
    lstm_probs = np.vstack(lstm_probs).squeeze()

    # TCN (8 features)
    tcn_dataset = SeqDataset(X_seq_test_8)
    tcn_loader = DataLoader(tcn_dataset, batch_size=128)
    tcn_probs = []
    with torch.no_grad():
        for xb in tcn_loader:
            xb = xb.to(device)
            tcn_probs.append(tcn(xb).cpu().numpy())
    tcn_probs = np.vstack(tcn_probs).squeeze()

    # ---- Combine predictions ----
    min_len = min(len(rf_probs), len(lgbm_probs), len(lstm_probs), len(tcn_probs))
    final_prob = (
        rf_probs[:min_len]
        + lgbm_probs[:min_len]
        + lstm_probs[:min_len]
        + tcn_probs[:min_len]
    ) / 4

    final_preds = (final_prob > 0.5).astype(int)
    y_true = y_test.values[:min_len]

    print(classification_report(y_true, final_preds, digits=4))



=== Hybrid Ensemble for target: oestrus (10% ADASYN) ===
              precision    recall  f1-score   support

           0     0.9736    0.9999    0.9865     54458
           1     0.1111    0.0007    0.0013      1478

    accuracy                         0.9735     55936
   macro avg     0.5424    0.5003    0.4939     55936
weighted avg     0.9508    0.9735    0.9605     55936


=== Hybrid Ensemble for target: calving (10% ADASYN) ===
              precision    recall  f1-score   support

           0     0.9749    0.9998    0.9872     54530
           1     0.3333    0.0036    0.0070      1406

    accuracy                         0.9748     55936
   macro avg     0.6541    0.5017    0.4971     55936
weighted avg     0.9588    0.9748    0.9626     55936


=== Hybrid Ensemble for target: lameness (10% ADASYN) ===
              precision    recall  f1-score   support

           0     0.9761    0.9999    0.9879     54597
           1     0.4545    0.0037    0.0074      1339

    acc